# Advanced document indexing

## Splitting and ingesting the content of a single URL (on Cornwall)

### Preparing the Chroma DB collections

In [4]:
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [5]:
ollama_embeddings = OllamaEmbeddings(model="bge-m3")

cornwall_granular_collection = Chroma(  # A
    collection_name="cornwall_granular",
    embedding_function=ollama_embeddings,
)

cornwall_granular_collection.reset_collection()  # B

# A Create a Chroma collection using local Ollama embeddings.
# B Reset the collection in case it already exists.

In [6]:
cornwall_coarse_collection = Chroma( # A 
    collection_name="cornwall_coarse",
    embedding_function=ollama_embeddings
)

cornwall_coarse_collection.reset_collection() # B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

### Loading the HTML content with the AsyncHtmlLoader

In [7]:
from langchain_community.document_loaders import AsyncHtmlLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()
len(docs)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.32it/s]


1

### Splitting into granular chunks with the HTMLSectionSplitter

In [9]:
from langchain_text_splitters import HTMLSectionSplitter

In [10]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks) 

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

granular_chunks = split_docs_into_granular_chunks(docs)

# Ingesting granular chunks
cornwall_granular_collection.add_documents(documents=granular_chunks)

# Searching granular chunks
results = cornwall_granular_collection.similarity_search(query="Events or festivals in Cornwall", k=3)
for doc in results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance 

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter

In [11]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=300)

def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A 
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

coarse_chunks = split_docs_into_coarse_chunks(docs)

# Ingesting coarse chunks
cornwall_coarse_collection.add_documents(documents=coarse_chunks)

# Searching coarse chunks
results = cornwall_coarse_collection.similarity_search(query="Events or festivals in Cornwall", k=3)
for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corni

## Splitting and ingesting the content of various URLs (across UK destinations)

In [13]:
# Preparing the Chroma DB collections
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=ollama_embeddings
)

uk_granular_collection.reset_collection() #B
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=ollama_embeddings
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter

In [27]:
# Reduce this list if you want to save on processing fees
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #C
    docs =  html_loader.load() #D
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists 
#C Loader for one destination
#D Documents of one destination

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.85it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.10it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.26it/s]


...

{'source': 'https://en.wikivoyage.org/wiki/Rye_(England)', 'title': 'Rye (England) – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.25it/s]


{'source': 'https://en.wikivoyage.org/wiki/Seaford', 'title': 'Seaford – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.00it/s]


{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest', 'title': 'Ashdown Forest – Travel guide at Wikivoyage', 'language': 'en'}


In [15]:
# Searching
granular_results = uk_granular_collection.similarity_search(query="Events or festivals in East Sussex", k=4)
for doc in granular_results:
    print(doc)

page_content='Do 
 [ edit ] 
 
 
 
 
   
 Brighton Theatres .   Brighton is a great place to see a theatre show or a gig. There are many many theatres and venues in and around Brighton.   
 
 
 
 50.81715 -0.12348 1   Sealanes  on Madeira Drive is an open-air lido with heated 50 m pool, opening in spring 2023. 
 
 Football:   
 
 50.8618 -0.0833 2   Brighton & Hove Albion ,   Falmer BN1 9BL   ( off A27 5   mi (8.0   km) northeast of the city by Falmer railway station ),   ☏   +44 1273 668855 .   "The Seagulls" play football in the Premier League, England's top tier, with their home ground at Falmer or Amex Stadium, capacity 30,750. Their women's team play in the Women's Super League, with home games at Broadfield Stadium in  Crawley , shared with Crawley Town. In 2025 Falmer Stadium hosted games in the Women's Rugby Union World Cup.         ( updated Sep 2025 ) 
 
 
 Cricket:   
 
 50.8306 -0.1645 3   Sussex CCC ,   Eaton Rd, Hove BN3 3AN   ( 1 mile west of central Brighton ),   ☏   +4

In [16]:
coarse_results = uk_coarse_collection.similarity_search(query="Events or festivals in East Sussex", k=4)
for doc in coarse_results:
    print(doc)

page_content='### Events

[edit]

A market during the Brighton Festival. The Theatre Royal is the red building.
A colourful parade down Queens Road during Pride in 2016.

  * **Brighton Racecourse** has flat-racing April-Oct. It's on Freshfield Rd a mile east of town centre.
  * **Plumpton Racecourse** is National Hunt (jumps races) Nov-March, but it's 10 mi (16 km) north in Lewes.
  * Brighton Festival Fringe: early May – early June, ☏ +44 1273 764900, info@brightonfringe.org. The Fringe runs at the same time as the main festival, and features over 600 events, including comedy, theatre, music, and "open houses" (local artists exhibiting in their own homes) and tours (haunted pubs, Regency Brighton, churches, cemeteries, sewers, etc.)_(date needs fixing)_
  * Brighton Festival: May, ☏ +44 1273 709709 (tickets), tickets@brightonfestival.org. The Brighton Festival, in May each year, is the second biggest arts festival in Great Britain (coming closely behind Edinburgh). Music of all sorts

In [17]:
granular_results = uk_granular_collection.similarity_search(query="Beaches in Cornwall", k=4)
for doc in granular_results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='North Cornwall' metadata={'Header 1': 'North Cornwall'}
page_content='West Cornwall' metadata={'Header 1': 'West Cornwall'}
page_content='South Cornwall' metadata={'Header 1': 'South Cornwall'}


In [18]:
coarse_results = uk_coarse_collection.similarity_search(query="Beaches in Cornwall", k=4)
for doc in coarse_results:
    print(doc)

page_content='**South Cornwall** is in Cornwall. It includes much of the stunning Cornish
coast along the English Channel of the Atlantic Ocean.

## Towns and villages

[edit]

Map of South Cornwall

  * 50.26-5.0511 Truro — Cornwall's main centre hosts the Royal Cornwall Museum
  * 50.3311-4.20212 Cawsand — overlooks Plymouth Sound; Cawsand is within Mount Edgcumbe Country Park
  * 50.15-5.073 Falmouth — famous for its beaches, it is home to the world's third largest natural harbour
  * 50.334-4.6334 Fowey — the Fowey Regatta in mid-August attracts many yachts and sailing boats
  * 50.354-4.4545 Looe — a summer resort place with a monkey sanctuary, and an active fishing village
  * 50.408-4.2126 Saltash — "Gateway to Cornwall", a small town on the Cornwall side of the Tamar crossings
  * 50.338-4.7957 St Austell — largest town in the county and home to the Eden Project, the world's largest greenhouse
  * 50.3314-4.75788 Charlestown — seaside town used as filming location for the TV sh

## Embedding strategy

### Embedding child chunks with ParentDocumentRetriever

In [28]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [29]:
ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [30]:
# Setting up the Parent Document retriever

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=ollama_embeddings,
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

In [31]:
# Ingesting the content into doc and vector store

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination 
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.82it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.20it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.68it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.28it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.05it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [32]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['2dfaf1e6-d012-4294-be35-d7853d8dc216',
 '09cba4d0-bd24-4ac1-a533-e3e4e80f83f4',
 'bf871fdc-572d-4a4c-9f8f-d9bc81792d0e',
 '9207a9d8-3c0c-4fe7-882c-884ee22e5610',
 '109622f2-f354-489c-844c-dac56dedbaa2',
 'f4441921-2134-4db3-9c94-1fc8a8db1490',
 '7639825f-6b9e-4f41-8e03-701fe8bf87a7',
 '9df8fe51-f6ca-4bab-90e1-be55fb56432d',
 'cc7599a3-e37f-486d-b042-ef9e7958eb17',
 'a2f60aa4-ebf5-4356-89f4-d32bd7d9ffc6',
 '40d6d989-bd55-4b18-bf28-ce866ea1a4b7',
 '0b184988-c110-4f48-b93d-8b6254d94248',
 '307223cc-d577-4476-a99a-8d71ac81716d',
 '4cb3df2b-efe6-4305-99a3-14fe57e9f745',
 'fee2a93b-5c6b-4659-a2af-d34f5bfd8368',
 '8728acd1-7d6a-4e73-a945-fab1e0473847',
 '8ab6ef80-88c5-4501-9bb3-61e166916e0e',
 '7e2019e5-c36a-4ce2-ab7b-d313575d802b',
 '011ea4e7-233a-4637-9976-bdf346bd76c6',
 '56fbb061-4100-45fa-9f9b-0191424bab85',
 'eecdffdd-8879-410d-9351-d2eee02f906f',
 '1359b2c5-7694-4adf-8294-babaaacf80fa',
 '922f5bcc-b6be-4e9d-8653-988486fe8115',
 '9abb838e-b53c-43e5-8fb2-994ef4c8fa1a',
 'ca63f11f-ef28-

In [34]:
# Performing a search on granular information

retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")
len(retrieved_docs)

4

In [35]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Penzance', 'title': 'Penzance – Travel guide at Wikivoyage', 'language': 'en'}, page_content='First Kernow bus T1 runs every 30 min between Penzance and Truro (1 hr 45\nmin), via St Erth, Hayle, Camborne, and Redruth. Change at Truro for Newquay,\nSt Austell and Bodmin. Reaching Plymouth and Exeter by bus is not worth the\nbother, take the train.\n\n### By car\n\n[edit]\n\nPenzance is a 5- to 6-hour drive from London via M4, M5, and A30. It\'s a long\nway and at some point you\'ll need to refuel. Don\'t be paying motorway prices,\nthere\'s supermarket petrol at (amongst others) M5 jcn 28 (Cullompton Tesco),\nA30 Bodmin (Asda, Launceston Rd Bodmin) and A30 Penzance (Tesco).\n\n### By boat\n\n[edit]\n\nA ferry plies between Penzance and the Isles of Scilly, daily from mid-March\nto October. The ferry (Scillonian III[dead link]) leaves Penzance around 9AM\nto reach the main island of St Mary\'s at noon; it returns at 4:30PM for\

In [36]:
# Comparing with direct semantic search on child chunks

child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")
print(len(child_docs_only))
child_docs_only[0]

4


Document(id='e9d9c071-2eb4-40ea-92ad-a33b6fbe975b', metadata={'title': 'Penzance – Travel guide at Wikivoyage', 'source': 'https://en.wikivoyage.org/wiki/Penzance', 'doc_id': '55c2d650-0240-4afe-9972-e17a2e00a05f', 'language': 'en'}, page_content='These "A"-buses, operated by First Kernow, are blue open-top double-deckers in\nsummer. For bus travel plus rail, a good deal is the _Ride Cornwall Ranger_\n(adult £13) described above. For bus only, buy a _Day Rider_ for £12 (child\n£6) from the Bus Station or from the driver on boarding - contactless bank\ncards accepted. Bus drivers also issue Ride Cornwall Rangers, but only for\nfull price, go to the station for concessions.\n\n## See\n\n[edit]')

### Embedding child chunks with MultiVectorRetriever